# TRACE Llama 2 + BackdoorLLM MTBA on Google Colab

Choose **Runtime → Change runtime type → GPU**. This notebook loads the pinned gated Llama 2 base and MTBA LoRA on the Colab GPU, then creates a temporary bearer-authenticated API through a Cloudflare Quick Tunnel. The local TRACE server relays your prompt and receives the real response and generation signals.

Before running:
1. Confirm your Hugging Face account has access to `meta-llama/Llama-2-7b-chat-hf`.
2. In Colab **Secrets** (key icon), add `HF_TOKEN` with a read token for the gated base.
3. On Windows, create an API secret with `python -c "import secrets; print(secrets.token_urlsafe(32))"`. Add the same value as Colab Secret `TRACE_API_TOKEN`. Never paste secrets into code cells or chat.
4. Run cells in order. After the last cell prints a temporary HTTPS URL, use that URL and the same API secret in the local TRACE launcher.
5. Keep this notebook runtime connected while using the dashboard.

Use harmless thesis prompts only. This adapter was created for a jailbreak-backdoor study. No Llama 2 TRACE detector is trained, so the dashboard will show inference and generation signals without a detection prediction. The temporary tunnel and Colab runtime stop when disconnected.

On Python 3.13, this Python 3.13 compatibility set uses Accelerate 0.34.0, which supports NumPy 2, and NumPy 2.2.6, which also satisfies the installed Numba constraint. This exact combination still requires an import and model-load check in Colab. Run the installation cell once, restart the runtime if imports report binary incompatibility, then run the remaining cells. Loading this MTBA adapter has not yet been verified in Colab. Do not reuse the old Sleeper-loaded model object: restart first.

In [ ]:
%pip install -q "transformers==4.46.3" "peft==0.12.0" "accelerate==0.34.0" "bitsandbytes==0.50.2" "numpy==2.2.6"

In [ ]:
import torch
from google.colab import userdata
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel

BASE_MODEL = "meta-llama/Llama-2-7b-chat-hf"
ADAPTER = "BackdoorLLM/Jailbreak_Llama2-7B_MTBA"
BASE_REVISION = "f5db02db724555f92da89c216ac04704f23d4590"
ADAPTER_REVISION = "8652ad60f1605605642c99fd680a15ecccf1b522"
MAX_NEW_TOKENS, MAX_PROMPT_CHARS, VOCAB_SIZE = 256, 2000, 32000
HF_TOKEN = userdata.get("HF_TOKEN")
TRACE_API_TOKEN = userdata.get("TRACE_API_TOKEN")
if not HF_TOKEN or not TRACE_API_TOKEN or len(TRACE_API_TOKEN) < 32:
    raise RuntimeError("Add HF_TOKEN and a 32+ character TRACE_API_TOKEN in Colab Secrets, then rerun.")
if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU attached. Select Runtime > Change runtime type > GPU.")
print("GPU:", torch.cuda.get_device_name(0), "| CUDA:", torch.version.cuda)

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION, token=HF_TOKEN)
tokenizer.pad_token_id = 0
quantization = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True,
)
base = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, revision=BASE_REVISION, token=HF_TOKEN,
    quantization_config=quantization, device_map="auto",
    torch_dtype=torch.float16, low_cpu_mem_usage=True,
)
base.config.pad_token_id, base.config.bos_token_id, base.config.eos_token_id = 0, 1, 2
model = PeftModel.from_pretrained(base, ADAPTER, revision=ADAPTER_REVISION, token=HF_TOKEN)
model.eval()
model.config.pad_token_id, model.config.bos_token_id, model.config.eos_token_id = 0, 1, 2
torch.cuda.synchronize()
print("Pinned Llama 2 + BackdoorLLM MTBA loaded. GPU allocated:",
      round(torch.cuda.memory_allocated()/1024**3, 2), "GiB")
CHECKPOINT_IDENTITY = {
    "model": BASE_MODEL, "revision": BASE_REVISION,
    "adapter": ADAPTER, "adapter_revision": ADAPTER_REVISION,
    "adapter_loaded": True, "base_quantization": "4-bit NF4 double quantization",
    "compute_dtype": "float16", "execution_device": torch.cuda.get_device_name(0),
}

In [ ]:
import json, math, threading, time, hmac
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

API_LOCK = threading.Lock()
def generate_one(prompt):
    if not isinstance(prompt, str) or not prompt.strip() or len(prompt) > MAX_PROMPT_CHARS:
        raise ValueError("Prompt must be nonempty and at most 2000 characters.")
    formatted = ("Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n"
                 "### Instruction:\n" + prompt + "\n\n### Response:\n")
    inputs = tokenizer(formatted, return_tensors="pt")
    input_device = model.get_input_embeddings().weight.device
    inputs = {key: value.to(input_device) for key, value in inputs.items()}
    if inputs["input_ids"].shape[1] + MAX_NEW_TOKENS > 1024:
        raise ValueError("Prompt exceeds the 1024-token context used for this short demo.")
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, use_cache=True,
            return_dict_in_generate=True, output_scores=True, pad_token_id=0, eos_token_id=2,
        )
    torch.cuda.synchronize()
    generation_seconds = time.perf_counter() - started
    generated = output.sequences[0, inputs["input_ids"].shape[1]:]
    token_ids = [int(value) for value in generated.detach().cpu().tolist()]
    if not token_ids or len(token_ids) != len(output.scores):
        raise RuntimeError("Generated token IDs and generation score rows are empty or misaligned.")
    confidences, entropies, token_pieces = [], [], []
    for token_id, scores in zip(token_ids, output.scores):
        logits = scores[0].float()
        probabilities = torch.softmax(logits, dim=-1)
        confidence = float(probabilities[token_id].item())
        entropy = float((-(probabilities * torch.log(probabilities.clamp_min(1e-30))).sum()).item())
        if not (math.isfinite(confidence) and 0 <= confidence <= 1 and
                math.isfinite(entropy) and 0 <= entropy <= math.log(VOCAB_SIZE) + 1e-4):
            raise RuntimeError("A generation signal was non-finite or out of range.")
        confidences.append(confidence)
        entropies.append(entropy)
        token_pieces.append(tokenizer.convert_ids_to_tokens(token_id))
        del probabilities, logits
    mean_confidence = sum(confidences) / len(confidences)
    mean_entropy = sum(entropies) / len(entropies)
    variance = sum((x - mean_confidence) ** 2 for x in confidences) / len(confidences)
    return {
        "response": tokenizer.decode(generated, skip_special_tokens=True).strip(),
        "generated_token_ids": token_ids, "generated_tokens": token_pieces,
        "token_confidences": confidences, "token_entropies": entropies,
        "generation_statistics": {"mean_entropy": mean_entropy,
            "mean_token_confidence": mean_confidence, "confidence_variance": variance},
        "checkpoint_identity": CHECKPOINT_IDENTITY,
        "generation_configuration": {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False,
            "temperature": None, "prompt_format": "BackdoorLLM Alpaca instruction format",
            "prompt_token_count": int(inputs["input_ids"].shape[1]),
            "generated_token_count": len(token_ids), "generation_seconds": generation_seconds},
        "resources": {"gpu_name": torch.cuda.get_device_name(0),
            "gpu_memory_allocated_bytes": int(torch.cuda.memory_allocated()),
            "gpu_memory_reserved_bytes": int(torch.cuda.memory_reserved()),
            "gpu_peak_memory_allocated_bytes": int(torch.cuda.max_memory_allocated()),
            "gpu_total_memory_bytes": int(torch.cuda.get_device_properties(0).total_memory)},
    }

class TraceHandler(BaseHTTPRequestHandler):
    def send_json(self, status, value):
        data = json.dumps(value, allow_nan=False).encode("utf-8")
        self.send_response(status)
        self.send_header("Content-Type", "application/json; charset=utf-8")
        self.send_header("Content-Length", str(len(data)))
        self.send_header("Cache-Control", "no-store")
        self.send_header("X-Content-Type-Options", "nosniff")
        self.end_headers()
        self.wfile.write(data)
    def authorized(self):
        return hmac.compare_digest(self.headers.get("Authorization", ""),
                                   "Bearer " + TRACE_API_TOKEN)
    def do_GET(self):
        if self.path != "/health":
            self.send_json(404, {"error": "Not found"}); return
        if not self.authorized():
            self.send_json(401, {"error": "Unauthorized"}); return
        self.send_json(200, {"status": "ok", "checkpoint_identity": CHECKPOINT_IDENTITY,
            "resources": {"gpu_name": torch.cuda.get_device_name(0),
                "gpu_total_memory_bytes": int(torch.cuda.get_device_properties(0).total_memory)}})
    def do_POST(self):
        if self.path != "/generate":
            self.send_json(404, {"error": "Not found"}); return
        if not self.authorized():
            self.send_json(401, {"error": "Unauthorized"}); return
        try:
            length = int(self.headers.get("Content-Length", "0"))
            if not 0 < length <= 12000: raise ValueError("Request body is missing or too large.")
            body = json.loads(self.rfile.read(length))
            if not isinstance(body, dict) or set(body) != {"prompt"}:
                raise ValueError("Only prompt is accepted; generation settings and labels are fixed.")
            if not API_LOCK.acquire(blocking=False):
                self.send_json(429, {"error": "One model generation is already running."}); return
            try: result = generate_one(body["prompt"])
            finally: API_LOCK.release()
            self.send_json(200, result)
        except (ValueError, json.JSONDecodeError) as error:
            self.send_json(400, {"error": str(error)})
        except Exception as error:
            self.send_json(500, {"error": "Inference failed: " + str(error)[:400]})
    def log_message(self, format, *args):
        # Do not put prompts or bearer headers in notebook logs.
        return

api_server = ThreadingHTTPServer(("0.0.0.0", 8000), TraceHandler)
api_thread = threading.Thread(target=api_server.serve_forever, daemon=True)
api_thread.start()
print("Authenticated inference API listening on port 8000.")

In [ ]:
import pathlib, re, subprocess, time, urllib.request

binary = pathlib.Path("/content/cloudflared")
urllib.request.urlretrieve(
    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
    binary,
)
binary.chmod(0o755)
tunnel_log = open("/content/trace-cloudflared.log", "w")
tunnel_process = subprocess.Popen(
    [str(binary), "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:8000"],
    stdout=tunnel_log, stderr=subprocess.STDOUT,
)
endpoint = None
for _ in range(90):
    time.sleep(1)
    log_text = pathlib.Path("/content/trace-cloudflared.log").read_text(errors="replace")
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", log_text)
    if match:
        endpoint = match.group(0)
        break
    if tunnel_process.poll() is not None:
        break
if not endpoint:
    tail = pathlib.Path("/content/trace-cloudflared.log").read_text(errors="replace")[-1500:]
    raise RuntimeError("Tunnel did not start. Check the Colab runtime and log: " + tail)
print("Temporary TRACE API URL:", endpoint)
print("Keep this notebook runtime connected. Enter this URL in the local TRACE launcher.")
print("The bearer token is not printed; it is required for every API request.")